In [5]:
#CISB5123 Text Analytics - Lab Assignment 1 (Web Scraping)

#Name  : Nik Amir Faris Bin Nik Amidi & Muhammad Thaqif bin Khairulanuar
#ID    : SW01083709 & SW01083763


import time
import csv
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.trustpilot.com/review/shopee.com.my"
MAX_PAGES = 5
OUTPUT_CSV = "reviews_scraped.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# HELPER FUNCTIONS

def fetch_page(url: str) -> str:
    """Send HTTP GET request and return HTML text."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.text

def parse_reviews(html: str) -> list[dict]:
    """
    Parse a Trustpilot review page HTML and extract:
    - reviewer name
    - review date
    - review content
    Returns list of dictionaries.
    """
    soup = BeautifulSoup(html, "html.parser")
    results = []

    # Each review is usually inside an <article> tag
    review_cards = soup.find_all("article")

    for card in review_cards:
        # Reviewer name (may vary slightly by layout)
        name_tag = card.select_one('[data-consumer-name-typography="true"]') or card.select_one("span.typography_heading-s__f7029")
        reviewer_name = name_tag.get_text(strip=True) if name_tag else "Unknown"

        # Review date (time tag)
        time_tag = card.find("time")
        review_date = time_tag.get("datetime") if time_tag and time_tag.get("datetime") else (time_tag.get_text(strip=True) if time_tag else "Unknown")

        # Review content
        content_tag = card.select_one('[data-service-review-text-typography="true"]') or card.select_one("p.typography_body-l__KUYFJ")
        review_content = content_tag.get_text(" ", strip=True) if content_tag else ""

        # Only keep rows that look like real reviews (non-empty content)
        if review_content:
            results.append({
                "reviewer_name": reviewer_name,
                "review_date": review_date,
                "review_content": review_content
            })

    return results

def save_to_csv(rows: list[dict], filename: str) -> None:
    """Save list of dict rows into a CSV file."""
    fieldnames = ["reviewer_name", "review_date", "review_content"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# MAIN SCRAPING LOGIC

all_reviews = []

for page in range(1, MAX_PAGES + 1):
    page_url = f"{BASE_URL}?page={page}"
    print(f"Scraping page {page}: {page_url}")

    html = fetch_page(page_url)
    page_reviews = parse_reviews(html)

    print(f"  Found {len(page_reviews)} reviews on page {page}")
    all_reviews.extend(page_reviews)

    # polite delay to reduce load / risk of blocking
    time.sleep(1)

# SAVE OUTPUT
save_to_csv(all_reviews, OUTPUT_CSV)
print(f"\nDone! Total reviews saved: {len(all_reviews)}")
print(f"Saved to: {OUTPUT_CSV}")

Scraping page 1: https://www.trustpilot.com/review/shopee.com.my?page=1
  Found 17 reviews on page 1
Scraping page 2: https://www.trustpilot.com/review/shopee.com.my?page=2
  Found 20 reviews on page 2
Scraping page 3: https://www.trustpilot.com/review/shopee.com.my?page=3
  Found 20 reviews on page 3
Scraping page 4: https://www.trustpilot.com/review/shopee.com.my?page=4
  Found 20 reviews on page 4
Scraping page 5: https://www.trustpilot.com/review/shopee.com.my?page=5
  Found 19 reviews on page 5

Done! Total reviews saved: 96
Saved to: reviews_scraped.csv
